In [5]:
import numpy as np
import pandas as pd

seed = 1337
rng = np.random.default_rng(seed)
N = 275000
chunkSize = 10000

columns = ["Global_SKU_Defect_Rate_%", "ABS_Volume_Difference", "Storage_Size", "Aisle_Hold_%", "#_Pick_Events", "#_Pick_Events_In_Clique",
           "#_Picks", "#_Picks_In_Clique", "Defect_In_Linked_Receive", "Time_In_Loc", "Current_Max_Volume"]



problemSKU = [True, False]
problemSKUProbs = np.array([0.1, 0.9])


#Parameters 
#Global_SKU_Defect_Rate_%
#Beta distribution 
defectAlpha, defectBeta = 2, 100

#ABS_Volume_Difference
#Normal distribution 
absVolumeMean = 0
absVolumeStd = 1000

#Storage_Size
#Assigned Distribution
storageSizes = ['size_1', 'size_2', 'size_3', 'size_4']
storageSizeProbs = np.array([0.4, 0.2, 0.3, 0.1])
#rng.choice(storageSizes, size=1000, p=storageSizeProbs)

#Aisle_Hold_%
#Beta distribution
aisleHoldAlpha, aisleHoldBeta = 0.425, 8.075

##_Pick_Events
#Poisson distribution
pickEventsLambda = 15

##_Pick_Events_In_Clique
#Poisson distribution
pickEventsInCliqueLambda = 100

##_Picks
#Poisson distribution
picksLambda = 35

##_Picks_In_Clique
#Poisson distribution
picksInCliqueLambda = 200

#Defect_In_Linked_Receive
#Assigned Distribution
defectInLinkedReceive = [True, False]
defectInLinkedReceiveProbs = np.array([0.05, 0.95])

#Time_In_Loc
#Negative Binomial distribution
timeInLocAlphaN = 9
timeInLocBetaP = 0.35

#Current_Max_Volume
#Normal distribution
currentMaxVolumeMean = 2160
currentMaxVolumeStd = 600



In [ ]:
def make_chunk(chunkSize, rng):
    # ----- 1. base covariates -----

    # Problem SKU mask
    problem_mask = rng.choice(
        [True, False], size=chunkSize, p=problemSKUProbs
    )

    # Global_SKU_Defect_Rate_% (as fraction 0–1)
    global_defect = rng.beta(defectAlpha, defectBeta, size=chunkSize)

    # ABS_Volume_Difference
    abs_vol = np.abs(rng.normal(absVolumeMean, absVolumeStd, size=chunkSize))

    # Storage_Size
    storage = rng.choice(storageSizes, size=chunkSize, p=storageSizeProbs)

    # Aisle_Hold_% (0–1)
    aisle_hold = rng.beta(aisleHoldAlpha, aisleHoldBeta, size=chunkSize)

    # Poisson counts
    pick_events = rng.poisson(pickEventsLambda, size=chunkSize)
    pick_events_clique = rng.poisson(pickEventsInCliqueLambda, size=chunkSize)
    picks = rng.poisson(picksLambda, size=chunkSize)
    picks_clique = rng.poisson(picksInCliqueLambda, size=chunkSize)

    # Defect_In_Linked_Receive (True/False)
    defect_linked = rng.choice(
        [True, False], size=chunkSize, p=defectInLinkedReceiveProbs
    )

    # Time_In_Loc ~ NB(n, p)
    time_in_loc = rng.negative_binomial(timeInLocAlphaN , timeInLocBetaP, size=chunkSize)
    time_in_loc = time_in_loc + 1  # to avoid log(0) later, shift by 1 day

    # Current_Max_Volume
    current_max_vol = rng.normal(
        currentMaxVolumeMean, currentMaxVolumeStd, size=chunkSize
    )
    current_max_vol = np.abs(current_max_vol)  # ensure non-negative

    # ----- 2. apply “problem SKU” effects (vectorized) -----
    # example: higher defect rate, higher aisle holds, more pick events for problem SKUs

    idx = problem_mask
    p_condition = 0.5  # probability each condition applies

    # Vectorized conditions (each applies independently to ~50% of problem SKUs)
    conds = [
        # (mask_condition, target_array, transform_func)
        (rng.random(chunkSize) < p_condition, global_defect, lambda x: np.clip(x * rng.uniform(1.5, 3.0, size=x.shape[0]), 0, 1)),
        (rng.random(chunkSize) < p_condition, abs_vol, lambda x: x * rng.uniform(1.2, 2.0, size=x.shape[0])),
        (rng.random(chunkSize) < p_condition, aisle_hold, lambda x: np.clip(x + rng.beta(1, 10, size=x.shape[0]), 0, 1)),
        (rng.random(chunkSize) < p_condition, pick_events, lambda x: x + rng.poisson(5.0, size=x.shape[0])),
        (rng.random(chunkSize) < p_condition, time_in_loc, lambda x: x + rng.negative_binomial(3, 0.5, size=x.shape[0])),
    ]

    for cond_mask_base, target, transform in conds:
        cond_mask = idx & cond_mask_base
        if cond_mask.sum() > 0:
            target[cond_mask] = transform(target[cond_mask])

    # ----- 3. DataFrame -----

    # Define your "true" logistic coefficients
    a0 = -5
    b_global = 5.0
    b_abs_vol = 0.002
    b_aisle = 1.5
    b_pick_events = 0.03
    b_pick_clique = 0.02
    b_time = 0.4
    b_max_vol = -0.0005

    # Safe logit_p computation
    safe_vars = {
        'global_defect': np.clip(global_defect, 0, 1),
        'abs_vol': abs_vol,
        'aisle_hold': np.clip(aisle_hold, 0, 1),
        'pick_events': pick_events,
        'pick_events_clique': pick_events_clique,
        'time_in_loc': np.maximum(time_in_loc, 1),
        'current_max_vol': np.maximum(current_max_vol, 1e-6),
    }

    
    
    logit_p = (
        a0
        + b_global * safe_vars['global_defect']
        + b_abs_vol * safe_vars['abs_vol']
        + b_aisle * safe_vars['aisle_hold']
        + b_pick_events * safe_vars['pick_events']
        + b_pick_clique * safe_vars['pick_events_clique']
        + b_time * np.log1p(safe_vars['time_in_loc'])
        + b_max_vol * np.log1p(safe_vars['current_max_vol'])
    )

    p_defect = np.clip(1 / (1 + np.exp(-logit_p)), 0.001, 0.999)  # logistic function
    defect = rng.binomial(1, p_defect)     # binary outcome

    # Now add to DataFrame
    df_chunk = pd.DataFrame({
        "Global_SKU_Defect_Rate_%": global_defect * 100.0,
        "ABS_Volume_Difference": abs_vol,
        "Storage_Size": storage,
        "Aisle_Hold_%": aisle_hold * 100.0,
        "#_Pick_Events": pick_events,
        "#_Pick_Events_In_Clique": pick_events_clique,
        "#_Picks": picks,
        "#_Picks_In_Clique": picks_clique,
        "Defect_In_Linked_Receive": defect_linked,
        "Time_In_Loc": time_in_loc,
        "Current_Max_Volume": current_max_vol,
        "Problem_SKU": problem_mask,
        "Defect": defect,           # binary target
        "Logit_Probability": logit_p,  # optional: true linear predictor
        "True_Prob_Defect": p_defect,  # optional: true probabilities
    })

    return df_chunk

import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
out_path = os.getenv("DATA_SAVE_DIR", "no_env_set")
out_path = f"{out_path}/synthetic_data_{timestamp}.csv"
print(out_path)
first = True
remaining = N

while remaining > 0:
    n_now = min(chunkSize, remaining)
    df = make_chunk(n_now, rng)

    df.to_csv(
        out_path,
        mode="w" if first else "a",
        header=first,
        index=False,
    )
    first = False
    remaining -= n_now

./SyntheticData/synthetic_data_2026_03_19_15_57_12.csv
